# Shared TF-IDF Feature Extraction

## Purpose
All TF-IDF models in this project use the **same fitted feature extractor**.

This notebook is the single source of truth for the TF-IDF representation.

The model notebooks load this saved vectorizer and call `.transform()` rather than fitting their own TF-IDF representation.

### 1. Imports and project paths

PyCharm runs the project notebooks from the `notebooks/` working directory, so the repository root is one level above it.

In [3]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path("../..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
TFIDF_MODEL_DIR = MODELS_DIR / "tfidf"

### 2. Load and validate the dataset splits

The existing split files remain the source data for every model. We verify the required columns and confirm that the same question ID does not appear across multiple splits.

In [7]:
train = pd.read_csv(PROCESSED_DIR / "train.csv")
validation = pd.read_csv(PROCESSED_DIR / "validation.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")

required = {"id", "source", "text", "target"}

for name, frame in (
    ("train", train),
    ("validation", validation),
    ("test", test),
):
    missing = required.difference(frame.columns)

    if missing:
        raise ValueError(
            f"{name} split is missing columns: {sorted(missing)}"
        )

    if frame["text"].isna().any() or frame["target"].isna().any():
        raise ValueError(
            f"{name} split contains missing text or target values"
        )

train_ids = set(train["id"])
validation_ids = set(validation["id"])
test_ids = set(test["id"])

assert not (
    train_ids & validation_ids
    or train_ids & test_ids
    or validation_ids & test_ids
)

for name, frame in (
    ("Train", train),
    ("Validation", validation),
    ("Test", test),
):
    print(
        f"{name}: {len(frame):,} answers; "
        f"{frame['id'].nunique():,} questions"
    )

Train: 58,823 answers; 16,706 questions
Validation: 12,611 answers; 3,580 questions
Test: 12,632 answers; 3,581 questions


### 3. Fit the shared TF-IDF vectorizer

The vectorizer is fit on the **training split only**. Validation and test data are transformed using the vocabulary learned from training.

This prevents information from validation or test text from influencing feature extraction and gives every TF-IDF classifier the same feature space.

In [8]:
VECTORIZER_CONFIG = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 100_000,
    "sublinear_tf": True,
    "strip_accents": "unicode",
    "dtype": np.float32,
}

vectorizer = TfidfVectorizer(**VECTORIZER_CONFIG)

X_train = vectorizer.fit_transform(
    train["text"].astype(str)
)

X_validation = vectorizer.transform(
    validation["text"].astype(str)
)

X_test = vectorizer.transform(
    test["text"].astype(str)
)

print("Train TF-IDF:", X_train.shape)
print("Validation TF-IDF:", X_validation.shape)
print("Test TF-IDF:", X_test.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Train TF-IDF: (58823, 100000)
Validation TF-IDF: (12611, 100000)
Test TF-IDF: (12632, 100000)
Vocabulary size: 100000


### 4. Save the fitted feature extractor

The fitted vectorizer stores both the TF-IDF configuration and the vocabulary learned from the training data.

Every TF-IDF model notebook should load this file instead of calling `fit()` or `fit_transform()` on a new vectorizer.

In [9]:
vectorizer_path = (
    TFIDF_MODEL_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(vectorizer, vectorizer_path)

print(f"Saved shared TF-IDF vectorizer to {vectorizer_path}")

Saved shared TF-IDF vectorizer to ../../models/tfidf/tfidf_vectorizer.joblib


## How model notebooks use this

A model notebook should now do:

```python
vectorizer = joblib.load(
    TFIDF_MODEL_DIR / "tfidf_vectorizer.joblib"
)

X_train = vectorizer.transform(train["text"].astype(str))
X_validation = vectorizer.transform(validation["text"].astype(str))
X_test = vectorizer.transform(test["text"].astype(str))
```

The CSV files remain the source dataset. The vectorizer is the saved **feature-extraction artifact**, so there is no need to commit large duplicate TF-IDF matrices for every model.